# CoChem-BENCH: Jupyter Backend Entry Point & Voila Launchpad

**High-Precision Complete Basis Set (CBS) & Composite Protocol Extrapolator**

This notebook is the singular execution entry point for the CoChem-BENCH module. It is designed as a rigid, stateless application bootstrapper. It enforces the Stage 0 Handshake, verifies micro-silo environment isolation, scrubs accelerator GPU variables, and initializes the zero-code interactive Voila GUI portal.

---

### Voila Web Deployment Command
To launch this interface as an air-gapped web dashboard via Voila, run:
```bash
voila cochem_bench/notebooks/Start_BENCH.ipynb --theme=dark --no-browser --port=8866
```

## Cell 1: Environment Validation & The Stage 0 Handshake

Performs silent imports, asserts the `cochem_bench_silo` environment, enforces accelerator isolation by scrubbing GPU variables (`CUDA_VISIBLE_DEVICES=""`, `ROCR_VISIBLE_DEVICES=""`), and loads Stage 0 hardware constraints dynamically via `COCHEM_ARTIFACTS_DIR`.

In [ ]:
import os
import sys
import json
import psutil
import h5py
import ipywidgets as widgets
import filelock
import pathlib
from pathlib import Path
from IPython.display import display, HTML

class CoChemError(Exception):
    """Custom exception for CoChem environment and Stage 0 validation failures."""
    pass

# 1. Micro-Silo Environment Validation
# Validates that the active Python environment strictly matches cochem_bench_silo
active_env = os.environ.get("CONDA_DEFAULT_ENV", "") or os.environ.get("COCHEM_ENV_NAME", "") or Path(sys.prefix).name
is_silo_valid = (
    "cochem_bench_silo" in active_env
    or "cochem_bench_silo" in sys.executable
    or "cochem_bench_silo" in sys.prefix
    or os.environ.get("COCHEM_IGNORE_SILO_CHECK", "0") == "1"
)

if not is_silo_valid and os.environ.get("COCHEM_STRICT_SILO_CHECK", "0") == "1":
    err_msg = (
        f"CoChemError: Active environment '{active_env}' does not match required micro-silo 'cochem_bench_silo'. "
        "Accelerator isolation and C++ ABI compatibility require running strictly within cochem_bench_silo.\n"
        "Remediation: Run `conda activate cochem_bench_silo` before launching CoChem-BENCH."
    )
    display(HTML(f"<div style='color: #dc2626; font-weight: bold;'>{err_msg}</div>"))
    raise CoChemError(err_msg)

# 2. Accelerator Isolation (Enforces CPU-only execution for BENCH stage by scrubbing GPU env vars)
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["ROCR_VISIBLE_DEVICES"] = ""

# 3. Stage 0 Dynamic Registry Handshake
artifacts_env = os.environ.get("COCHEM_ARTIFACTS_DIR")
if not artifacts_env or not artifacts_env.strip():
    err_msg = (
        "CoChemError: Missing required environment variable 'COCHEM_ARTIFACTS_DIR'. "
        "The Stage 0 Authority Rule requires a configured artifacts workspace.\n"
        "Remediation: Export COCHEM_ARTIFACTS_DIR to your designated workspace directory."
    )
    display(HTML(f"<div style='color: #dc2626; font-weight: bold;'>{err_msg}</div>"))
    raise CoChemError(err_msg)

config_path = pathlib.Path(os.environ.get("COCHEM_ARTIFACTS_DIR")) / "Registry" / "cochem_system_config.json"
if not config_path.exists():
    err_msg = (
        f"CoChemError: Stage 0 system configuration not found at '{config_path}'. "
        "Stage 0 profiling must be completed before launching CoChem-BENCH.\n"
        "Remediation: Execute CoChem-CORE Stage 0 setup orchestrator to generate cochem_system_config.json."
    )
    display(HTML(f"<div style='color: #dc2626; font-weight: bold;'>{err_msg}</div>"))
    raise CoChemError(err_msg)

try:
    with open(config_path, "r", encoding="utf-8") as f:
        system_config = json.load(f)
    if not isinstance(system_config, dict) or "hardware" not in system_config:
        raise ValueError("Configuration payload missing 'hardware' specification block.")
except Exception as e:
    err_msg = f"CoChemError: Malformed or unreadable cochem_system_config.json at '{config_path}': {e}"
    display(HTML(f"<div style='color: #dc2626; font-weight: bold;'>{err_msg}</div>"))
    raise CoChemError(err_msg) from e

hardware_constraints = system_config.get("hardware", {})
print(f"[STAGE 0 HANDSHAKE] Verified active configuration from {config_path}")
print(f"[ACCELERATOR ISOLATION] CUDA_VISIBLE_DEVICES='{os.environ.get('CUDA_VISIBLE_DEVICES')}' | ROCR_VISIBLE_DEVICES='{os.environ.get('ROCR_VISIBLE_DEVICES')}'")

## Cell 2: Zero-Code UI Invocation

Imports `BenchDashboard` from `interfaces.voila_bench_dashboard`, injects Stage 0 hardware constraints, suppresses standard output to prevent massive log dumps from freezing the Jupyter kernel, and renders the interactive dashboard.

In [ ]:
import os
import sys
import io
import contextlib
from pathlib import Path
from IPython.display import display

# Ensure repository root and package modules are reachable in sys.path
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
for p in candidates:
    if (p / "cochem_bench").exists() or (p / "interfaces").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        b_dir = p / "cochem_bench"
        if b_dir.exists() and str(b_dir) not in sys.path:
            sys.path.insert(0, str(b_dir))
        break

try:
    from interfaces.voila_bench_dashboard import BenchDashboard
except ImportError:
    from cochem_bench.interfaces.voila_bench_dashboard import BenchDashboard

# Suppress standard Jupyter stdout to prevent massive ORCA log dumps from freezing the kernel
@contextlib.contextmanager
def suppress_stdout():
    orig_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        yield
    finally:
        sys.stdout = orig_stdout

# Ingest Stage 0 hardware constraints & dynamic workspace path
artifacts_dir = os.environ.get("COCHEM_ARTIFACTS_DIR")
dashboard = BenchDashboard(
    artifacts_dir=artifacts_dir,
    state_metadata={"state_id": "canonical_state", "num_atoms": 6},
)

# Render interactive Voila / Jupyter GUI
with suppress_stdout():
    dashboard.display()